# Функция потерь и градиентный спуск — простыми словами

Учебный ноутбук: **что такое Loss**, чем он отличается от метрики, и как модель **учится**, спускаясь по «склону» ошибки.

Разбираем:
- Loss vs метрика;
- **MSE**, **MAE**, **Log Loss** как функции потерь;
- **градиентный спуск** (одна переменная → много параметров);
- линейная регрессия и логистическая регрессия «изнутри».

**Что понадобится:** `numpy`, `matplotlib`, `scikit-learn`.

Запускайте ячейки **сверху вниз**.

---

## План

1. [Словарь](#dict)
2. [Что такое функция потерь](#loss)
3. [Loss ≠ метрика](#loss-vs-metric)
4. [MSE как Loss](#mse)
5. [MAE как Loss](#mae)
6. [Log Loss (Binary Cross Entropy)](#logloss)
7. [Градиентный спуск: аналогия с горой](#gd)
8. [Формула обновления и learning rate](#update)
9. [Частные производные и градиент](#partial)
10. [GD для линейной регрессии (код)](#linreg-gd)
11. [GD для логистической регрессии (код)](#logreg-gd)
12. [Что реально делает `fit()` в sklearn](#sklearn)
13. [Шпаргалка](#итог)

> **Заметка о точности:** в тексте ниже исправлены и явно помечены частые упрощения (например, `LinearRegression.fit` в sklearn **не** всегда = градиентный спуск).


<a id="dict"></a>
## 1. Словарь

| Термин | Простыми словами |
|--------|------------------|
| **Параметры / коэффициенты** | Числа модели: $w$, $b$ (веса и сдвиг), которые **подбираем** |
| **Предсказание $\hat{y}$** | То, что модель выдала сейчас |
| **Функция потерь (Loss)** | Число «насколько плохо» **на текущих** параметрах (для **обучения**) |
| **Метрика** | Число «насколько хорошо» для **человека** (часто после обучения) |
| **Производная $dL/dw$** | Наклон Loss по одному параметру |
| **Частная производная $\partial L/\partial w_j$** | Наклон, когда параметров много: меняем **только** $w_j$ |
| **Градиент $\nabla L$** | Вектор **всех** частных производных |
| **Градиентный спуск** | Шаги **против** градиента, чтобы Loss **уменьшить** |
| **Learning rate $\eta$** | Размер шага |
| **Эпоха / итерация** | Один (или серия) проход обновления параметров |
| **Сигмоида** | $p = 1/(1+e^{-z})$ — число $z$ → вероятность в $(0,1)$ |


<a id="loss"></a>
## 2. Что такое функция потерь (Loss Function)?

**Функция потерь** — это число, которое показывает:

> **Насколько плохо** модель предсказала данные **на текущем шаге** обучения.

**Главная цель обучения** (для многих моделей):

> **Минимизировать** функцию потерь.

### Пример с квартирой

| | |
|--|--|
| Истина $y$ | 10 000 000 ₽ |
| Предсказание $\hat{y}$ | 9 800 000 ₽ |
| Ошибка $y - \hat{y}$ | 200 000 ₽ |

Loss **превращает** ошибку в одно число.  
Если взять **квадрат** ошибки (как в MSE на одном объекте):

$$
(200\,000)^2 = 40\,000\,000\,000
$$

Модель «понимает»: ошиблась → нужно **изменить** $w$, $b$.

### Грубая схема обучения

```text
1. Задать начальные коэффициенты (случайные / нули / …)
2. Сделать предсказания
3. Посчитать Loss
4. Понять, как поправить коэффициенты (градиент)
5. Обновить коэффициенты
6. Повторить много раз, пока Loss почти не перестанет падать
```

Весь смысл — **сжимать Loss**.


<a id="loss-vs-metric"></a>
## 3. Loss ≠ метрика

| | **Loss** | **Метрика** |
|--|----------|-------------|
| Когда | **Во время** обучения | Часто **после** / для отчёта |
| Кому нужна | **Алгоритму** (оптимизатору) | **Человеку** / бизнесу |
| Примеры | MSE, Log Loss, Cross Entropy | $R^2$, Accuracy, F1, ROC AUC |
| Свойства | Желательно **гладкая**, дифференцируемая | Может быть «скачкообразной» |

Некоторые величины бывают **и** loss, **и** метрикой (MSE, MAE, Log Loss).  
Но **F1 / Accuracy** как loss почти не используют.

### Почему нельзя «учиться по Accuracy»?

Истина: класс **1**.

| Модель | $p$ (класс 1) | Класс при пороге 0.5 | Accuracy |
|--------|---------------|----------------------|----------|
| A | 0.51 | 1 | верно |
| B | 0.99 | 1 | верно |

Accuracy **одинаковая**, а модель B **гораздо** лучше.  
Для обучения нужно знать **насколько** ошиблись.

**Log Loss** (при $y=1$): $L = -\ln(p)$

- $p=0.51$ → $L \approx 0.67$  
- $p=0.99$ → $L \approx 0.01$  

Loss **различает** уверенность.

### Почему не F1 / Precision / Recall как Loss?

Нужно ответить: «чуть сдвинул $w$ — стало **лучше или хуже**?»  
Для этого функция должна меняться **плавно**.

Accuracy/F1 зависят от **жёсткого** класса после порога:

```text
p = 0.49 → класс 0
чуть сдвинули → p = 0.50 → класс 1
метрика скачком изменилась
```

По скачку **нельзя** надёжно взять производную и «направление шага».  
Loss по **вероятностям** (0.49 → 0.50 → 0.51) гладкий → работает градиентный спуск.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Accuracy не различает 0.51 и 0.99; Log Loss — различает
for p in [0.51, 0.99, 0.01]:
    y = 1
    acc = int(p >= 0.5)  # «угадали класс?»
    logloss = -np.log(p)
    print(f"y=1, p={p:.2f} → accuracy_hit={acc}, log_loss={logloss:.3f}")

# Плавность Log Loss vs скачок Accuracy
ps = np.linspace(0.01, 0.99, 200)
y = 1
ll = -np.log(ps)
acc = (ps >= 0.5).astype(float)

fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
ax[0].plot(ps, ll)
ax[0].set_title("Log Loss при y=1 (гладко)")
ax[0].set_xlabel("p"); ax[0].set_ylabel("Loss")
ax[0].grid(True, alpha=0.3)

ax[1].plot(ps, acc, drawstyle="steps-post")
ax[1].set_title("Accuracy hit (скачок на 0.5)")
ax[1].set_xlabel("p"); ax[1].set_ylabel("угадал класс?")
ax[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


<a id="mse"></a>
## 4. MSE как функция потерь

Модель: $\hat{y} = wx + b$ (или с несколькими признаками).

$$
\mathrm{MSE} = \frac{1}{m}\sum_{i=1}^{m}(\hat{y}^{(i)} - y^{(i)})^2
$$

($m$ — число объектов; иногда пишут $n$.)

Пока $w$, $b$ «плохие», MSE большое → цель: **уменьшить MSE**.

### Почему MSE популярна

1. **Сильнее** штрафует большие ошибки (бонус).  
2. **Главное:** график MSE по параметрам линейной модели — **гладкая «чаша»** (выпуклая).  
   В каждой точке есть производная → градиентный спуск «знает», куда идти.

### Почему не «сумма ошибок со знаком»

Ошибки $+5$ и $-5$ → сумма **0**, хотя модель ошиблась.  
Квадрат (или модуль) знаки убирает.

### Уточнение

«MSE всегда идеальна» — **нет**.  
При **выбросах** огромные квадраты могут **перетянуть** модель. Тогда смотрят MAE / Huber.

Но для **классической** линейной регрессии MSE — стандарт: и как loss, и как путь к **аналитическому** решению (МНК).


In [ ]:
# MSE vs «сумма ошибок со знаком»
errors = np.array([5.0, -5.0, 2.0, -2.0])
print("Ошибки:", errors)
print("Сумма со знаком (плохая идея как loss):", errors.sum())
print("MSE:", np.mean(errors ** 2))
print("MAE:", np.mean(np.abs(errors)))

# Чаша MSE для ŷ = w*x (один параметр), данные y ≈ 2x
x = np.array([1.0, 2.0, 3.0, 4.0])
y = 2.0 * x + np.array([0.1, -0.1, 0.05, -0.05])
ws = np.linspace(0, 4, 200)
mse_curve = [np.mean((w * x - y) ** 2) for w in ws]

plt.figure(figsize=(6, 3.5))
plt.plot(ws, mse_curve)
plt.axvline(2.0, color="red", ls="--", label="примерно истинный w≈2")
plt.xlabel("w"); plt.ylabel("MSE"); plt.title("Loss(w) — гладкая «чаша»")
plt.legend(); plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()


<a id="mae"></a>
## 5. MAE как функция потерь

$$
\mathrm{MAE} = \frac{1}{m}\sum_{i=1}^{m}\lvert \hat{y}^{(i)} - y^{(i)} \rvert
$$

Обучение по смыслу то же: предсказания → MAE → правим параметры → снова.

### MSE vs MAE на двух наборах ошибок

| | Ошибки | MAE | MSE |
|--|--------|-----|-----|
| Модель A | 1, 1, 1, 1 | 1 | 1 |
| Модель B | 0, 0, 0, 4 | 1 | **4** |

MAE: «модели одинаковы».  
MSE: «B намного хуже» (одна крупная ошибка).

### Плюс MAE

**Менее чувствительна к выбросам** — часто лучше, если в данных «дикие» точки.

### Минус для классического GD

График $|u|$ в нуле — **излом**.  
Обычная производная:

- слева от 0: $-1$  
- справа: $+1$  
- **в 0 не существует**

Значит «в самом дне» классический градиент **не определён**.  
Оптимизировать **можно** (субградиенты, другие методы, бустинг с `loss="absolute_error"`, нейросети с MAE), но **сложнее**, чем с MSE.

### Что чаще

| Ситуация | Loss |
|----------|------|
| Классическая линейная регрессия | **MSE** |
| Много выбросов | **MAE** или **Huber** |

> MSE выбрали не потому что «всегда точнее», а потому что **удобно оптимизировать**.  
> MAE часто **уместнее по смыслу** при выбросах.


In [ ]:
# |u| — излом в нуле; MSE — гладкая
u = np.linspace(-2, 2, 401)
plt.figure(figsize=(8, 3.2))
plt.subplot(1, 2, 1)
plt.plot(u, u ** 2)
plt.title("u² (как вклад в MSE) — гладко")
plt.grid(True, alpha=0.3)
plt.subplot(1, 2, 2)
plt.plot(u, np.abs(u))
plt.title("|u| (как вклад в MAE) — угол в 0")
plt.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

print("Производная |u|: слева -1, справа +1, в 0 — не определена (субградиент ∈ [-1,1]).")


<a id="logloss"></a>
## 6. Log Loss (Binary Cross Entropy) как Loss

Для **одного** объекта ($y \in \{0,1\}$, $p$ — вероятность класса 1):

$$
L = -\bigl[ y\,\ln p + (1-y)\,\ln(1-p) \bigr]
$$

- если $y=1$: $L = -\ln p$  
- если $y=0$: $L = -\ln(1-p)$  

Пока $p$ чуть неверна — Loss умеренный.  
Если модель **уверена и неправа** ($y=1$, $p\to 0$) — Loss → **$+\infty$**.

### Пример

$y=1$, $p=0.2$ → $L = -\ln(0.2) \approx 1.61$  

Если после обновления весов $p=0.8$ → $L = -\ln(0.8) \approx 0.22$  

Loss **упал** — шаг был в хорошую сторону.

### Почему Log Loss для логистической регрессии

1. Работает с **вероятностями**  
2. **Гладкая** (в рабочей области $p\in(0,1)$) → есть градиент  
3. **Сильно** бьёт за уверенные ошибки  

Логрегрессия **учится вероятностям** (`predict_proba`).  
Класс — уже **после** порога (часто 0.5).

### Цикл `fit` (схема)

```text
коэффициенты → z → сигмоида → p → Log Loss → градиент → новые коэффициенты → …
```


In [ ]:
# Log Loss при y=1: штраф за маленькую p
ps = np.linspace(0.01, 0.99, 200)
plt.figure(figsize=(6, 3.5))
plt.plot(ps, -np.log(ps), label="y=1: -ln(p)")
plt.plot(ps, -np.log(1 - ps), label="y=0: -ln(1-p)")
plt.xlabel("p"); plt.ylabel("Log Loss")
plt.title("Уверенная ошибка → огромный Loss")
plt.legend(); plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()

print("y=1, p=0.2 →", -np.log(0.2))
print("y=1, p=0.8 →", -np.log(0.8))


<a id="gd"></a>
## 7. Градиентный спуск: представь гору

Ночь, склон, **ничего не видно**.  
Знаешь только **наклон** земли под ногами.

Что делать? **Маленький шаг вниз** → снова почувствовать наклон → ещё шаг…  
Так доходишь до **низины**.

Это и есть **градиентный спуск**.

### Связь с ML

| Гора | ML |
|------|-----|
| Твоё положение | Параметры $w$, $b$ |
| Высота | **Loss** |
| Наклон под ногами | **Производная / градиент** |
| Шаг вниз | Обновление параметров |

Для MSE от одного $w$ «гора» часто похожа на **параболу-чашу**.  
Цель: $w$, при котором Loss **минимален**.

### Куда идти? Смотрим на наклон (производную)

Производная $dL/dw$ — **наклон** в точке.

| Наклон | Loss растёт… | Шаг градиентного спуска |
|--------|----------------|-------------------------|
| **> 0** | вправо (при росте $w$) | **уменьшить** $w$ (влево) |
| **< 0** | влево | **увеличить** $w$ (вправо) |
| **= 0** | — | стоим у **экстремума** (для чаши MSE — минимум) |

> Уточнение: «наклон = 0 ⇒ глобальный минимум» верно для **выпуклых** задач вроде MSE линейной регрессии.  
> В нейросетях бывают **локальные** минимумы и плато — упрощённо мы говорим «около хорошей точки».


<a id="update"></a>
## 8. Главная формула обновления

$$
w_{\mathrm{new}} = w_{\mathrm{old}} - \eta \cdot \frac{dL}{dw}
$$

| Символ | Смысл |
|--------|--------|
| $w$ | коэффициент |
| $\eta$ (eta) | **learning rate** — размер шага |
| $dL/dw$ | производная Loss по $w$ |

### Зачем минус?

Градиент указывает, куда Loss **растёт**.  
Нам нужно **спускаться** → идём **против** градиента.

- $dL/dw = +5$ → Loss растёт при увеличении $w$ → вычитаем → $w$ ↓  
- $dL/dw = -5$ → Loss растёт при уменьшении $w$ → минус на минус → $w$ ↑  
- $dL/dw = 0$ → $w$ не меняется

### Learning rate

| $\eta$ слишком мал | $\eta$ слишком велик |
|--------------------|------------------------|
| Ползём вечность | Перепрыгиваем дно, Loss скачет, может **разойтись** |

### Внутри одной итерации (идея `fit` при GD)

1. Текущие коэффициенты  
2. Предсказания  
3. Loss  
4. Производные / градиент  
5. Обновление коэффициентов  
6. Повтор

**Три действия по сути:** предсказать → оценить ошибку (Loss) → шаг против градиента.


In [ ]:
# Иллюстрация: спуск по параболе L(w) = (w - 3)^2
def L(w):
    return (w - 3.0) ** 2

def dL_dw(w):
    return 2.0 * (w - 3.0)

def gd_path(w0, eta, steps=15):
    w = w0
    hist = [w]
    for _ in range(steps):
        w = w - eta * dL_dw(w)
        hist.append(w)
    return np.array(hist)

ws = np.linspace(-1, 7, 200)

fig, axes = plt.subplots(1, 3, figsize=(11, 3.2), sharey=True)
for ax, eta, title in zip(
    axes,
    [0.1, 0.02, 0.9],
    ["η=0.1 — нормально", "η=0.02 — медленно", "η=0.9 — слишком большой шаг"],
):
    path = gd_path(0.0, eta, steps=15)
    ax.plot(ws, L(ws), color="steelblue")
    ax.plot(path, L(path), "o-", color="crimson", markersize=4)
    ax.set_title(title)
    ax.set_xlabel("w")
    ax.grid(True, alpha=0.3)
axes[0].set_ylabel("L(w)")
plt.suptitle("Градиентный спуск по L=(w-3)² — роль learning rate")
plt.tight_layout()
plt.show()

print("Финал при η=0.1:", gd_path(0.0, 0.1, 30)[-1], "(истина минимума w=3)")


<a id="partial"></a>
## 9. Частные производные и градиент

### Зачем частная производная?

Одна переменная: $L(w)$ → обычная $dL/dw$.

Много параметров: $\hat{y} = w_1 x_1 + \ldots + w_n x_n + b$  
→ Loss $L(w_1,\ldots,w_n,b)$.

Нужно знать отдельно:

- как Loss меняется при изменении **только** $w_1$;  
- только $w_2$;  
- только $b$.

Это **частные производные**: $\partial L/\partial w_1$, $\partial L/\partial w_2$, $\partial L/\partial b$.

Знак $\partial$: «переменных много, но сейчас двигаем **одну**, остальные **фиксируем**».

### Пример

$$
L(w_1, w_2) = w_1^2 + 3 w_2^2
$$

$$
\frac{\partial L}{\partial w_1} = 2w_1, \quad \frac{\partial L}{\partial w_2} = 6w_2
$$

В точке $(w_1, w_2) = (2, 1)$:

$$
\nabla L = [4,\, 6]
$$

Loss быстрее растёт по $w_2$ → при спуске $w_2$ меняем **сильнее**.

### Градиент

$$
\nabla L = \Bigl[\tfrac{\partial L}{\partial w_1},\,\ldots,\,\tfrac{\partial L}{\partial w_n},\,\tfrac{\partial L}{\partial b}\Bigr]
$$

- $\nabla L$ — направление **самого крутого подъёма** Loss  
- спуск: шаг в сторону **$-\nabla L$**

### Обновление

$$
w_j \leftarrow w_j - \eta \frac{\partial L}{\partial w_j}, \quad
b \leftarrow b - \eta \frac{\partial L}{\partial b}
$$

Векторно: $\theta \leftarrow \theta - \eta\,\nabla L$.

### Числовой шаг

$\eta = 0.1$, старт $(2, 1)$, $\nabla L = [4, 6]$:

$$
w_1 \leftarrow 2 - 0.1\cdot 4 = 1.6, \quad w_2 \leftarrow 1 - 0.1\cdot 6 = 0.4
$$

$$
L(2,1)=7 \;\rightarrow\; L(1.6, 0.4)=3.04
$$


In [ ]:
# Числовой пример из текста
def L(w1, w2):
    return w1 ** 2 + 3 * w2 ** 2

w1, w2 = 2.0, 1.0
eta = 0.1
print(f"Старт: w1={w1}, w2={w2}, L={L(w1,w2):.2f}")

for step in range(1, 6):
    g1, g2 = 2 * w1, 6 * w2  # градиент
    w1 = w1 - eta * g1
    w2 = w2 - eta * g2
    print(f"Шаг {step}: w1={w1:.4f}, w2={w2:.4f}, L={L(w1,w2):.4f}, grad=[{g1:.2f}, {g2:.2f}]")

print("→ ползём к (0,0), где L=0 (минимум).")


### Формулы градиента (усреднённый Loss)

**Линейная регрессия + MSE**  
$\hat{y}^{(i)} = w^\top x^{(i)} + b$,  
$L = \frac{1}{m}\sum_i (\hat{y}^{(i)} - y^{(i)})^2$

$$
\frac{\partial L}{\partial w_j} = \frac{2}{m}\sum_i (\hat{y}^{(i)} - y^{(i)})\, x^{(i)}_j, \quad
\frac{\partial L}{\partial b} = \frac{2}{m}\sum_i (\hat{y}^{(i)} - y^{(i)})
$$

**Логистическая регрессия + Log Loss (среднее по объектам)**  
$z^{(i)} = w^\top x^{(i)} + b$,  
$p^{(i)} = \sigma(z^{(i)}) = 1/(1+e^{-z^{(i)}})$

$$
\frac{\partial L}{\partial w_j} = \frac{1}{m}\sum_i (p^{(i)} - y^{(i)})\, x^{(i)}_j, \quad
\frac{\partial L}{\partial b} = \frac{1}{m}\sum_i (p^{(i)} - y^{(i)})
$$

Для **одного** объекта множитель $1/m$ пропадает:  
$\partial L/\partial w = (p-y)\,x$, $\partial L/\partial b = p-y$.

Если $y=1$, $p=0.2$, то $p-y = -0.8$ — вероятность **маловата**, обновление толкнёт параметры так, чтобы $p$ **выросла** (при правильном знаке $x$ и минусе в формуле спуска).

### Картина обучения

```text
w, b  →  предсказания (ŷ или p)  →  Loss  →  ∇L  →  θ ← θ − η∇L  →  снова…
```


<a id="linreg-gd"></a>
## 10. Градиентный спуск для линейной регрессии (код)

Соберём вручную: данные → MSE → градиент → цикл обновлений → сравним с `LinearRegression` sklearn.


In [ ]:
import numpy as np
from sklearn.linear_model import LinearRegression

np.random.seed(42)
# Истина: y = 3x + 2 + шум
m = 40
X = np.linspace(0, 5, m).reshape(-1, 1)
y = (3 * X[:, 0] + 2 + np.random.normal(0, 0.4, size=m))

def predict(X, w, b):
    return X[:, 0] * w + b

def mse(y_true, y_hat):
    return np.mean((y_hat - y_true) ** 2)

def gradients_mse(X, y, w, b):
    # L = mean( (y_hat - y)^2 )  →  dL/dw = 2*mean( (y_hat-y)*x ), dL/db = 2*mean(y_hat-y)
    y_hat = predict(X, w, b)
    err = y_hat - y
    dw = 2 * np.mean(err * X[:, 0])
    db = 2 * np.mean(err)
    return dw, db, mse(y, y_hat)

def train_linreg_gd(X, y, eta=0.05, steps=200, w0=0.0, b0=0.0):
    w, b = w0, b0
    history = []
    for t in range(steps):
        dw, db, loss = gradients_mse(X, y, w, b)
        w = w - eta * dw
        b = b - eta * db
        history.append(loss)
    return w, b, np.array(history)

w_gd, b_gd, hist = train_linreg_gd(X, y, eta=0.08, steps=250)
print(f"GD:     w={w_gd:.3f}, b={b_gd:.3f}, final MSE={hist[-1]:.4f}")

# sklearn LinearRegression — аналитическое МНК (не GD!), для сравнения «ответа»
lr = LinearRegression().fit(X, y)
print(f"sklearn: w={lr.coef_[0]:.3f}, b={lr.intercept_:.3f}, MSE={mse(y, lr.predict(X)):.4f}")
print("(Должны быть близки: GD сошёлся почти к тому же минимуму MSE.)")

plt.figure(figsize=(10, 3.5))
plt.subplot(1, 2, 1)
plt.plot(hist)
plt.xlabel("итерация"); plt.ylabel("MSE"); plt.title("Loss падает по ходу GD")
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.scatter(X[:, 0], y, s=20, label="данные")
xs = np.linspace(0, 5, 50)
plt.plot(xs, w_gd * xs + b_gd, "r-", label=f"GD: ŷ={w_gd:.2f}x+{b_gd:.2f}")
plt.plot(xs, lr.coef_[0] * xs + lr.intercept_, "g--", label="sklearn OLS")
plt.legend(); plt.title("Прямая после обучения"); plt.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()


<a id="logreg-gd"></a>
## 11. Градиентный спуск для логистической регрессии

Та же схема, но между $z = wx + b$ и Loss стоит **сигмоида**.

### Шаги

1. **Старт** $w$, $b$ (нули или малый random)  
2. **$z = wx + b$** (ещё не вероятность; может быть $-5$, $100$…)  
3. **$p = \sigma(z) = 1/(1+e^{-z})$**  
   - $z=0$ → $p=0.5$  
   - $z=2$ → $p\approx 0.88$  
   - $z=-2$ → $p\approx 0.12$  
4. **Log Loss**  
5. **Градиент** $(p-y)$ …  
6. **$w \leftarrow w - \eta\,\partial L/\partial w$** и то же для $b$  
7. Повтор, пока Loss почти не стабилизируется  

$e \approx 2.71828$ — основание натурального логарифма.


In [ ]:
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss, accuracy_score

def sigmoid(z):
    # стабильная сигмоида
    z = np.clip(z, -500, 500)
    return 1.0 / (1.0 + np.exp(-z))

# --- сигмоида ---
zs = np.linspace(-6, 6, 200)
plt.figure(figsize=(6, 3))
plt.plot(zs, sigmoid(zs))
plt.axhline(0.5, color="gray", ls="--")
plt.title("Сигмоида: z → вероятность")
plt.xlabel("z"); plt.ylabel("p"); plt.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

for z in [0, 2, -2]:
    print(f"z={z:2d} → p={sigmoid(z):.3f}")

# --- данные ---
np.random.seed(0)
X_c, y_c = make_classification(
    n_samples=200, n_features=1, n_informative=1, n_redundant=0,
    n_clusters_per_class=1, flip_y=0.05, class_sep=1.5, random_state=0
)
# для простоты один признак
x = X_c[:, 0]

def train_logreg_gd(x, y, eta=0.5, steps=400):
    w, b = 0.0, 0.0
    m = len(y)
    history = []
    for _ in range(steps):
        z = w * x + b
        p = sigmoid(z)
        # средний Log Loss
        eps = 1e-12
        loss = -np.mean(y * np.log(p + eps) + (1 - y) * np.log(1 - p + eps))
        # градиенты среднего Log Loss
        dw = np.mean((p - y) * x)
        db = np.mean(p - y)
        w = w - eta * dw
        b = b - eta * db
        history.append(loss)
    return w, b, np.array(history)

w_l, b_l, hist_l = train_logreg_gd(x, y_c, eta=0.8, steps=500)
p_hat = sigmoid(w_l * x + b_l)
y_hat = (p_hat >= 0.5).astype(int)

print(f"\nGD logreg: w={w_l:.3f}, b={b_l:.3f}")
print(f"  log_loss={log_loss(y_c, p_hat):.4f}, accuracy={accuracy_score(y_c, y_hat):.3f}")

# sklearn (итеративный солвер, не обязательно «ванильный» GD)
clf = LogisticRegression(solver="lbfgs").fit(X_c, y_c)
p_sk = clf.predict_proba(X_c)[:, 1]
print(f"sklearn:   w={clf.coef_[0,0]:.3f}, b={clf.intercept_[0]:.3f}")
print(f"  log_loss={log_loss(y_c, p_sk):.4f}, accuracy={accuracy_score(y_c, clf.predict(X_c)):.3f}")

plt.figure(figsize=(10, 3.5))
plt.subplot(1, 2, 1)
plt.plot(hist_l)
plt.xlabel("итерация"); plt.ylabel("Log Loss"); plt.title("Loss падает (наш GD)")
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
# один объект: y=1, p от 0.05 до 0.95 — как градиент (p-y) толкает
ps = np.linspace(0.05, 0.95, 50)
plt.plot(ps, ps - 1, label="y=1: (p-y)")
plt.plot(ps, ps - 0, label="y=0: (p-y)")
plt.axhline(0, color="k", lw=0.8)
plt.xlabel("p"); plt.ylabel("p - y"); plt.title("Знак (p−y) = направление ошибки")
plt.legend(); plt.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

print("\nЕсли y=1 и p=0.2, то p-y=-0.8 → после шага w,b вероятность обычно растёт.")


<a id="sklearn"></a>
## 12. Что реально делает `fit()` в sklearn? (важно!)

В учебных схемах часто говорят: «внутри `fit` — градиентный спуск».  
**Идея верная** для многих моделей, но **детали зависят от класса**.

| Модель sklearn | Как обычно ищет минимум Loss |
|----------------|------------------------------|
| `LinearRegression` | **Не GD.** Аналитическое МНК / least squares (закрытая формула) |
| `SGDRegressor` / `SGDClassifier` | Стохастический градиентный спуск |
| `LogisticRegression` | Итеративные солверы (`lbfgs`, `liblinear`, `saga`…) — **оптимизация** Log Loss, но не всегда «ванильный» batch-GD |
| Нейросети (PyTorch и т.д.) | Почти всегда варианты GD/Adam |

**Что общего:**

1. Есть параметры  
2. Есть Loss  
3. Параметры двигают так, чтобы Loss **уменьшался**  

GD — самый понятный способ это объяснить «на пальцах».

### Полный цикл (ещё раз)

```text
текущие w₁…wₙ, b
        ↓
   предсказания ŷ или p
        ↓
   функция потерь L
        ↓
   частные производные → градиент ∇L
        ↓
   θ ← θ − η ∇L
        ↓
   следующий шаг
```


<a id="итог"></a>
## 13. Самое главное + шпаргалка

### Loss

1. Loss = «насколько плохо **сейчас**» для **обучения**.  
2. Метрика = «насколько хорошо» для **оценки человеком**.  
3. Учимся на **гладких** loss (MSE, Log Loss), а не на скачущих Accuracy/F1.

### Популярные Loss

| Задача | Loss |
|--------|------|
| Регрессия | MSE, MAE, Huber |
| Бинарная классификация | Log Loss (BCE) |
| Много классы | Cross Entropy |

### Градиентный спуск

1. Производная / градиент = направление **роста** Loss.  
2. Шаг: $\theta \leftarrow \theta - \eta\nabla L$.  
3. $\eta$ мал — медленно; велик — можно разнести обучение.  
4. Много параметров → **частные** производные → вектор-градиент.  
5. Линейная регрессия: MSE + градиент $(2/m)\sum(\hat{y}-y)x$.  
6. Логрегрессия: сигмоида + Log Loss + градиент $(1/m)\sum(p-y)x$.

### MSE vs MAE (как loss)

| | MSE | MAE |
|--|-----|-----|
| Выбросы | сильно бьёт | мягче |
| Гладкость | везде | излом в 0 |
| Классический GD | удобно | субградиенты / другие методы |

### Три действия внутри обучения

1. Предсказать  
2. Посчитать Loss  
3. Шаг **против** градиента  

Повторять, пока Loss почти не перестанет уменьшаться (или кончится лимит итераций).


### Мини-практика

1. В GD для линейной регрессии поставьте `eta=2.0` — что случится с `hist`?  
2. Поставьте `eta=0.001` и мало `steps` — дойдёт ли до sklearn-коэффициентов?  
3. Для логрегрессии выведите Log Loss при $y=1$ для $p\in\{0.9, 0.5, 0.1\}$.  
4. Объясните одной фразой: почему Accuracy плохой loss, а Log Loss — хороший.


In [ ]:
# ===== Мини-практика: подсказки / быстрые опыты =====
print("1) Слишком большой η для параболы L=(w-3)^2:")
w = 0.0
for t in range(8):
    w = w - 2.0 * (2 * (w - 3))  # eta=2, dL/dw=2(w-3) — разнос
    print(f"  step {t+1}: w={w:.3f}, L={(w-3)**2:.3f}")

print("\n3) Log Loss y=1:")
for p in [0.9, 0.5, 0.1]:
    print(f"  p={p} → L={-np.log(p):.3f}")

print("\n4) Accuracy не видит разницу 0.51 vs 0.99; Log Loss — видит и гладкая.")


## Что делать дальше

1. Свяжите с ноутбуками **метрик**: метрика — отчёт; loss — двигатель `fit`.  
2. Свяжите с **валидацией**: Loss/метрику на train смотрят для обучения; выбор модели — по valid/test.  
3. В нейросетях та же идея, но Loss сложнее, а оптимизатор часто **Adam** (умнее, чем голый GD).

### Главная мысль

> Обучение — это **повторяющееся уменьшение Loss**  
> маленькими шагами **против градиента**.  
> Сигмоида и Log Loss делают то же для **вероятностей** в классификации.

Удачи!
